# ROG-Dialog v2 — explorer

Interactive look at ROG-Dialog v2: length / WPS distributions, duration × WPS sentiment topography, and random listens across duration bands with transcript, gold sentiment, WPS, and per-matrix-cell predictions when the instance is in the 2 s-filtered test split.

**Kernel:** select the `ssp` mamba env in VS Code.

Plot and loader code lives in `utils_rog_explorer.py` so this notebook stays scannable. Nothing is written to disk; nothing is retrained.

In [ ]:
import json
import random
import sys
from pathlib import Path

from IPython.display import Audio, Markdown, display

ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'data' / 'rog_dialog_v2_instance.jsonl').exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError('Could not locate repo root with data/rog_dialog_v2_instance.jsonl')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'scripts' / '6_sent_duration'))
import utils_rog_explorer as u
print('ROOT =', ROOT)

ROG_PATH   = ROOT / 'data' / 'rog_dialog_v2_instance.jsonl'
MATRIX_DIR = ROOT / 'misc' / 'run_results' / 'finetune_20261113_162724'

random.seed(42)

In [ ]:
df = u.load_rog(ROG_PATH)
cell_preds = u.load_matrix_predictions(MATRIX_DIR)
df.head(3)

## Distributions — duration, words, WPS

Also writes `docs/img/ROG-dia-v2_histograms.png` so the dataset-overview page stays in sync.

In [ ]:
fig = u.plot_histograms(df, duration_floor=2.0)
out_png = ROOT / 'docs' / 'img' / 'ROG-dia-v2_histograms.png'
out_png.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out_png, dpi=150, bbox_inches='tight')
print(f"<2s: {(df['duration'] < 2).sum()} ({100*(df['duration']<2).mean():.1f}%)")
print(f"2-5s: {((df['duration'] >= 2) & (df['duration'] < 5)).sum()}")
print(f"5-10s: {((df['duration'] >= 5) & (df['duration'] < 10)).sum()}")
print(f">10s: {(df['duration'] >= 10).sum()}")

## Sentiment topography — 2-D heatmap over duration × WPS

Where in the (duration, WPS) plane does ROG's mean sentiment ordinal drift away from `neutralPositive` (ord 3, ROG's modal class)? Below 3 is negative-leaning, above 3 is positive-leaning. Each cell shows the mean ordinal and the sample count.

In [ ]:
fig = u.plot_sentiment_topography_2d(df, n_dur_bins=5, n_wps_bins=5, wps_clip=10.0, mode='quantile')

## Sentiment topography — 3-D surface

Same data, surface view. Useful for eyeballing whether there is a *ridge* of positive or negative sentiment along either axis, versus a flat landscape with noise.

In [ ]:
fig = u.plot_sentiment_topography_3d(df, n_dur_bins=5, n_wps_bins=5, wps_clip=10.0, mode='quantile')

## Listen to random utterances per duration band

Re-run a cell to draw a new sample. Model-performance grid only populates for instances in the 2 s-filtered ROG test split (n=434).

In [ ]:
def listen(sub_df, n=5, seed=None):
    sampled = u.random_sample(sub_df, n=n, seed=seed)
    print(f"seed = {sampled.attrs.get('seed')}  · pool size = {len(sub_df)}")
    for i, (_, row) in enumerate(sampled.iterrows(), 1):
        display(Markdown(
            f"### {i}. `{row['instance_id']}`  —  **{row['sentiment']}**  (3-cls: {row['sentiment_3']})\n"
            f"- split: `{row['split']}`  · speaker: `{row['speaker']}`\n"
            f"- duration: **{row['duration']:.2f} s**  ·  words: **{row['words']}**  ·  WPS: **{row['wps']:.2f}**\n"
            f"- transcript: *{row['text']}*"
        ))
        wav = ROOT / row['audio_path'] if row['audio_path'] else None
        if wav is not None and wav.exists():
            display(Audio(str(wav)))
        else:
            display(Markdown(f'_audio missing at {wav}_'))
        display(Markdown(u.matrix_grid_md(row['instance_id'], cell_preds)))

### `< 2 s`

In [ ]:
listen(df[df['duration'] < 2].reset_index(drop=True), n=5)

### `2 – 5 s`

In [ ]:
listen(df[(df['duration'] >= 2) & (df['duration'] < 5)].reset_index(drop=True), n=5)

### `5 – 10 s`

In [ ]:
listen(df[(df['duration'] >= 5) & (df['duration'] < 10)].reset_index(drop=True), n=5)

### `> 10 s`

In [ ]:
listen(df[df['duration'] >= 10].reset_index(drop=True), n=5)